# Output Parser
**Output Parser**는 대규모 언어 모델(LLM, Large Language Model)의 출력 결과를 **애플리케이션에서 활용할 수 있도록 적절한 형식으로 변환**하는 도구이다.
- LLM은 일반적으로 텍스트 형태로 응답을 생성하지만, 이 텍스트는 그대로 활용하기 어려운 경우가 많다.
- Output Parser는 이러한 **비구조적 텍스트 데이터를 구조화된 데이터로 변환**하여 프로그램에서 활용 가능하도록 만든다.
- 예를 들어, 키워드 리스트를 뽑거나 JSON 형식으로 정보를 변환하는 데 사용된다.

## 주요 Output Parser 종류

1. **CommaSeparatedListOutputParser**
   - 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
   - 예: `"사과, 바나나, 포도"` → `["사과", "바나나", "포도"]`
2. **JsonOutputParser**
   - LLM의 출력이 JSON 형식일 때 이를 Python의 `dict` 객체로 변환한다.
   - JSON(JavaScript Object Notation)은 데이터 구조를 표현하기 위한 경량 포맷이다.
3. **PydanticOutputParser**
   - JSON 데이터를 Python의 [Pydantic](https://docs.pydantic.dev) 모델로 변환한다.
   - Pydantic은 데이터 유효성 검사와 설정 관리에 널리 사용되는 Python 라이브러리이다.
4. **StrOutputParser**
   - 모델의 출력 결과를 단순 문자열로 반환한다.
   - Chat 기반 모델은 Message 객체의 속성으로 LLM 결과를 반환한다. 거기에서 응답 문자열만 추출해서 반환한다.
> `JsonOutputParser`, `PydanticOutputParser` 는 모두 Pydantic을 사용해 데이터 구조(schema)를 정의하고, 해당 구조에 따라 출력을 검증하고 변환한다.

## 주요 메소드
- `parse(text: str)`
  - LLM이 생성한 문자열 응답을 받아 정해진 구조로 변환하여 반환한다.
- `get_format_instructions() -> str`
  - 각 OutputParer가 변환할 수있는 형식으로 LLM이 응답하도록 하는 프롬프트 텍스트를 반환한다.
  - 이 내용을 프롬프트에 넣어서 LLM이 정확한 포맷으로 응답하도록 유도한다.
  
## 참고
- Output Parser는 일반적으로 [`Runnable`](05_chaing_LECL.ipynb#Runnable) 인터페이스를 상속하여 구현되며, `invoke()` 메서드를 통해 실행할 수 있다.
- `invoke()`는 내부적으로 `parse()`를 호출하여 동작한다.
- 필요한 경우 Output Parser를 직접 구현하여 사용자 정의 출력 포맷을 처리할 수도 있다. 


## StrOutputParser
- 모델(LLM)의 출력 결과를 string으로 변환하여 반환하는 output parser.
- Chat Model은  Message 객체에서 content 속성값을 추출하여 문자열로 반환한다.

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI 
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
prompt = ChatPromptTemplate.from_template(
    template="한국의 {topic}에 관련된 속담 {count}개를 알려줘. 니가 만들지 말고 실제 있는 속담을 알려줘. 목록형식으로 출력해줘."
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

# prompt-[Query]->model-[response:AIMessage]->parser-[문자열]->최종결과
query = prompt.invoke({"topic":"호랑이", "count":3})
print(query)
response = model.invoke(query)
response

messages=[HumanMessage(content='한국의 호랑이에 관련된 속담 3개를 알려줘. 니가 만들지 말고 실제 있는 속담을 알려줘. 목록형식으로 출력해줘.', additional_kwargs={}, response_metadata={})]


AIMessage(content='1. **호랑이에게 물려 가도 정신만 차리면 산다**  \n2. **호랑이 굴에 가야 호랑이를 잡는다**  \n3. **호랑이도 제 말하면 온다**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 44, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Dto2TUH3c5pDgguc5dGjws5y7d0Xk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef2f7-60e6-7fe2-902f-d31781fac90a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 54, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
res_str = parser.invoke(response)
print(res_str)
print(response.content)

1. **호랑이에게 물려 가도 정신만 차리면 산다**  
2. **호랑이 굴에 가야 호랑이를 잡는다**  
3. **호랑이도 제 말하면 온다**
1. **호랑이에게 물려 가도 정신만 차리면 산다**  
2. **호랑이 굴에 가야 호랑이를 잡는다**  
3. **호랑이도 제 말하면 온다**


## CommaSeparatedListOutputParser

- 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
  - "a,b,c" => ['a','b','c']

In [25]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
txt = "서울,인천,부산,광주,대구,대전,울산"
# txt = "찾은 도시이름은 서울과 인천과 대전입니다."
r1 = parser.parse(txt)
r2 = parser.invoke(txt)

print(r1)
print(r2)

['서울', '인천', '부산', '광주', '대구', '대전', '울산']
['서울', '인천', '부산', '광주', '대구', '대전', '울산']


In [13]:
# Output Parser에 맞는 출력을 LLM 모델이 하도록 프롬프트에 넣을 지침을 조회.
instruction = parser.get_format_instructions()
print(instruction)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [17]:
ChatPromptTemplate?

Init signature:
ChatPromptTemplate(
    messages: 'Sequence[MessageLikeRepresentation]',
    *,
    template_format: 'PromptTemplateFormat' = 'f-string',
    name: str | None = None,
    input_variables: list[str],
    optional_variables: list[str] = [],
    input_types: dict[str, typing.Any] = <factory>,
    output_parser: langchain_core.output_parsers.base.BaseOutputParser | None = None,
    partial_variables: collections.abc.Mapping[str, typing.Any] = <factory>,
    metadata: dict[str, typing.Any] | None = None,
    tags: list[str] | None = None,
    validate_template: bool = False,
) -> None
Docstring:     
Prompt template for chat models.

Use to create flexible templated prompts for chat models.

!!! example

    ```python
    from langchain_core.prompts import ChatPromptTemplate

    template = ChatPromptTemplate(
        [
            ("system", "You are a helpful AI bot. Your name is {name}."),
            ("human", "Hello, how are you doing?"),
            ("ai", "I'm doing w

In [ ]:
prompt = ChatPromptTemplate(
    messages=[
        {"role":"system", "content":"출력형식: {output_format}"},
        {"role":"user", "content":"{query}"}
    ],
    partial_variables={"output_format":instruction} # input_variables에 값을 프롬프트템플릿 생성하면서 입력한다.
)

model = ChatOpenAI(model="gpt-5.4-nano")

query = prompt.invoke({"query":"자동차 종류 다섯가지를 알려줘."})
response = model.invoke(query)

In [24]:
res = parser.invoke(response)
print(res)



['세단', 'SUV', '해치백', '쿠페', '픽업트럭']


## JsonOutputParser

- JSON 형식의 응답을 dictionary로 반환한다.
- JSON 형식을 정하려는 경우 [Pydantic](Ref_typing_Pydantic.ipynb)을 이용해 JSON 스키마를 정의하여 JsonOutputParser 생성시 전달한다.
  - Pydantic 모델클래스를 이용해 LLM 모델이 응답할 때 json의 어떤 key에 어떤 응답을 작성할 지 Field로 정의한다.
  - Schema 지정은 필수는 아니다. 
- LLM이 JSON Schema를 따르는 형태로 응답을 하면 JsonOutputParser는 Dictionary로 변환한다.

In [30]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
# LLM 출력 형식 프롬프트
print(parser.get_format_instructions())

txt = '{"name":"이순신", "age":30, "address":"서울"}'
r1 = parser.parse(txt)
r2 = parser.invoke(txt)

print(type(r1), r1)
print(type(r2), r2)

Return a JSON object.
<class 'dict'> {'name': '이순신', 'age': 30, 'address': '서울'}
<class 'dict'> {'name': '이순신', 'age': 30, 'address': '서울'}


In [ ]:
parser = JsonOutputParser()
prompt = ChatPromptTemplate(
    messages=[
        ("system", "output format: {output_format}"),
        ("user", "{query}")
    ],
    partial_variables={"output_format":parser.get_format_instructions()}
)
model = ChatOpenAI(model="gpt-5.4-mini")

query = prompt.invoke({"query":"이순신 장군에 대해서 알려줘."})
# query.messages
res = model.invoke(query)

In [40]:
print(res.content)
# final_answer = parser.invoke(res)

이순신 장군(1545~1598)은 조선의 명장이자 한국 역사에서 가장 존경받는 인물 중 한 명입니다. 임진왜란 때 조선 수군을 이끌고 일본군을 상대로 큰 승리를 거두며 나라를 지켜낸 인물로 유명합니다.

### 간단한 소개
- **출생**: 1545년, 한양
- **사망**: 1598년, 노량해전 중 전사
- **주요 역할**: 조선 수군 총지휘관
- **대표 업적**: 한산도 대첩, 명량해전, 노량해전 등에서 승리

### 왜 유명한가?
이순신 장군은 단순히 전쟁에서 이긴 장수가 아니라,
- 뛰어난 **전략가**
- 철저한 **준비와 훈련**
- 강한 **책임감과 리더십**
- 불리한 상황에서도 포기하지 않는 **의지**

로 많은 존경을 받습니다.

### 대표적인 해전
1. **옥포해전**
   - 임진왜란 초기, 조선 수군이 처음으로 큰 승리를 거둔 전투 중 하나입니다.

2. **한산도 대첩**
   - “학익진” 전법으로 유명합니다.
   - 조선 수군이 일본군의 해상 보급로를 크게 무너뜨렸습니다.

3. **명량해전**
   - 매우 적은 수의 배로 수십 배 많은 적을 상대로 승리한 전투입니다.
   - 이순신의 리더십과 전술이 가장 극적으로 드러난 사례입니다.

4. **노량해전**
   - 마지막 전투로, 승리를 눈앞에 두고 전사했습니다.

### 성격과 평가
이순신은 매우 엄격하고 원칙적인 성품으로 알려져 있습니다. 군율을 엄격하게 지키게 했고, 백성을 생각하는 마음도 깊었습니다. 전쟁 중에도 백성을 보호하고 군대의 기강을 세우는 데 힘썼습니다.

### 유명한 말
가장 널리 알려진 말 중 하나는:

- **“아직 신에게는 열두 척의 배가 있사옵니다.”**
  
명량해전 직전 절망적인 상황에서 남긴 말로 전해집니다.

원하시면 제가 다음 중 하나로 더 자세히 설명해드릴 수 있어요:
1. **이순신 장군의 생애**
2. **임진왜란과 해전 이야기**
3. **이순신의 리더십**
4. **초등학생도 이해하기 쉽게 정리**


In [38]:
final_answer['answer']
final_answer.keys()

dict_keys(['answer'])

In [43]:
# JSON 형식에 대한 스키마 설계
## 어떤 키를 가지며, 그 키에 어떤 값을 넣어야 하는지를 설계. => Pydantic 모델을 이용
from pydantic import BaseModel, Field 
class PersonInfoSchema(BaseModel):
    """
    인물 정보에 대한 응답을 위한 Schema
    """
    # 변수명(key): 결과값의 타입 = Field(description="어떤 값을 넣을지 설명")
    name: str = Field(description="조회한 사람의 이름")
    yob: int = Field(description="조회한 사람의 출생 년도. 모를 경우는 -1을 대입")
    yod: int = Field(description="조회한 사람의 사망 년도. 모를 경우는 -1을 대입")
    profile: str = Field(description="조회한 사람의 주요 업적 소개")

parser2 = JsonOutputParser(pydantic_object=PersonInfoSchema)
# parser.get_format_instructions()
print(parser2.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [61]:
prompt2 = ChatPromptTemplate(
    messages=[
        ("system", "output format: {output_format}"),
        ("user", "{query}")
    ],
    partial_variables={"output_format":parser2.get_format_instructions()}
)

query2 = prompt2.invoke({"query":"고려 척준경장군 대해 알려줘."})
res2 = model.invoke(query2)

In [62]:
# res2.content
answer = parser2.invoke(res2)
answer#['name']

{'name': '척준경',
 'yob': -1,
 'yod': -1,
 'profile': '고려 시대의 무신으로, 여진 정벌과 대외 전투에서 활약한 장군이다. 특히 윤관의 별무반에 참여해 북방 여진 세력과의 전투에서 공을 세운 것으로 알려져 있으며, 강한 전투력과 무용으로 이름을 떨쳤다. 다만 정치적으로는 무신 정권 수립기 이전 인물로, 이후 혼란한 고려 무신 정치의 전개와는 구분된다.'}

## PydanticOutputParser

- JSON 형태로 받은 응답을 Pydantic 모델로 변환하여 반환한다.
- 구현은 JsonOutputParser와 동일한데 parsing 결과를 pydantic 모델타입으로 반환한다.

In [1]:
from pydantic import BaseModel, Field 
class PersonInfoSchema(BaseModel):
    """
    인물 정보에 대한 응답을 위한 Schema
    """
    # 변수명(key): 결과값의 타입 = Field(description="어떤 값을 넣을지 설명")
    name: str = Field(description="조회한 사람의 이름")
    yob: int = Field(description="조회한 사람의 출생 년도. 모를 경우는 -1을 대입")
    yod: int = Field(description="조회한 사람의 사망 년도. 모를 경우는 -1을 대입")
    profile: str = Field(description="조회한 사람의 주요 업적 소개")

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI 

parser = PydanticOutputParser(pydantic_object=PersonInfoSchema)
# print(parser.get_format_instructions())
prompt = ChatPromptTemplate(
    messages = [
        ("system", "당신은 초등학교 역사선생님입니다. 다음 출력 형식에 맞춰 답변해주세요.\n\n<출력형식>{output_format}</출력형식>"),
        ("user", "{query}")
    ],
    partial_variables={"output_format":parser.get_format_instructions()}
)
model = ChatOpenAI(model="gpt-5.4-mini")

In [ ]:

# 1. Prompt 생성
query = prompt.invoke({"query":query})
# print(query)

# 2. model에게 요청
res = model.invoke(query)
# print(type(res), res)

# 3. 응답을 응답 포멧에 맞게 parsing 
final_answer = parser.invoke(res)


In [14]:
print(type(final_answer))
print(final_answer)

<class '__main__.PersonInfoSchema'>
name='세종대왕' yob=1397 yod=1450 profile='조선의 네 번째 왕으로, 백성을 사랑하고 나라를 잘 다스린 훌륭한 임금입니다. 한글(훈민정음)을 만들도록 하여 우리 백성이 쉽게 글을 읽고 쓸 수 있게 했고, 과학·농업·음악·천문 분야도 크게 발전시켰습니다.'


In [16]:
print(final_answer.name)
print(final_answer.yob, final_answer.yod)
print(final_answer.profile)

세종대왕
1397 1450
조선의 네 번째 왕으로, 백성을 사랑하고 나라를 잘 다스린 훌륭한 임금입니다. 한글(훈민정음)을 만들도록 하여 우리 백성이 쉽게 글을 읽고 쓸 수 있게 했고, 과학·농업·음악·천문 분야도 크게 발전시켰습니다.


# LLM모델에 출력 형식을 설정

- ChatModel객체의 `with_structured_output(pydantic.BaseModel)` 을 이용해 모델의 출력 형식을 모델 자체에 추가할 수있다.
- `OutputParser`는 모델의 출력 결과를 받아서 형식을 변경해 준다. 그래서 Chain에 탈/부착을 통해 형식을 적용하거나 적용하지 않는 것을 자유롭게 할 수있다.
- 모델의 출력 결과를 항상 일정하게 할 경우에는 아예 **모델에 출력 형식을 설정할 수 있다.**

In [ ]:
model_with_output = model.with_structured_output(PersonInfoSchema)
res = model_with_output.invoke("세종대왕에 대해서 설명해주세요.") # 출력형식이 설정된 모델을 호출

# model.invoke("세종대왕에 대해서 설명해주세요.") # 출력형식이 설정 안된 모델을 호출.

In [21]:
res
# res.name

PersonInfoSchema(name='세종대왕', yob=1397, yod=1450, profile='조선의 제4대 왕으로, 훈민정음을 창제하고 과학·천문·농업·음악·의학 등 여러 분야의 발전을 이끈 군주입니다. 백성을 위한 통치와 학문 진흥으로 한국 역사에서 가장 위대한 왕 중 한 명으로 평가됩니다.')

In [22]:
res = model_with_output.invoke("AI Agent에 특징과 장단점에 대해서 정리해서 설명해줘.")

In [23]:
res

PersonInfoSchema(name='AI Agent', yob=-1, yod=-1, profile='AI Agent는 사용자의 목표를 이해하고, 필요한 작업을 스스로 계획·실행하며, 도구와 외부 시스템을 활용해 결과를 만들어내는 인공지능 시스템이다. 단순히 질문에 답하는 챗봇과 달리, 여러 단계의 작업을 이어서 처리하고 상황에 따라 행동을 조정할 수 있다는 점이 특징이다.\n\n특징:\n- 목표 지향성: 사용자가 원하는 최종 목표를 기준으로 작업을 수행함\n- 자율성: 일부 또는 상당 부분을 스스로 판단해 다음 행동을 결정함\n- 도구 활용: 검색, 코드 실행, 데이터 조회, API 호출 등 외부 도구와 연동 가능\n- 단계적 계획: 큰 문제를 여러 단계로 나누어 처리함\n- 적응성: 중간 결과나 새로운 정보에 따라 계획을 수정할 수 있음\n- 기억/상태 관리: 이전 대화나 작업 맥락을 반영해 연속적인 작업 수행이 가능함\n\n장점:\n- 반복 업무 자동화에 유리함\n- 복잡한 작업을 단계별로 처리할 수 있음\n- 다양한 도구를 조합해 효율을 높일 수 있음\n- 사람의 시간을 절약하고 생산성을 높일 수 있음\n- 24시간 작동 가능해 지속적인 업무 처리에 강함\n\n단점:\n- 잘못된 판단을 내릴 수 있어 오류가 누적될 수 있음\n- 목표나 지시가 모호하면 엉뚱한 결과를 낼 수 있음\n- 도구 연동이 많아질수록 보안, 권한, 개인정보 이슈가 커짐\n- 비용과 운영 복잡도가 높아질 수 있음\n- 완전한 자율성은 아직 한계가 있어 사람의 검토가 필요한 경우가 많음\n\n정리하면, AI Agent는 단순 응답형 AI보다 훨씬 능동적으로 일을 처리할 수 있는 시스템이지만, 그만큼 정확성, 안전성, 관리 측면에서 주의가 필요하다.')

# Streaming 방식 응답 처리

- Streaming 방식 응답 처리란, LLM이 텍스트를 모두 생성할 때까지 기다리지 않고, 생성되는 즉시 **부분적인 결과**를 실시간으로 전달받아 처리하는 방식을 의미한다. 이는 사용자가 응답을 더 빠르게 인지할 수 있게 해 주며, 특히 대화형 서비스, 실시간 UI 출력, 긴 문서 생성과 같은 상황에서 매우 유용하게 활용된다.

- `invoke()` 요청으로 받는 응답은 **비 스트리밍 방식**으로 모든 응답 텍스트 생성이 완료된 이후 그 결과를 한 번에 반환하는 구조이다. 반면 Streaming 방식은 **토큰(token) 단위 또는 여러 토큰이 묶인 청크(chunk) 단위**로 연속적인 데이터 스트림을 전송한다는 점에서 큰 차이가 있다. 즉, Streaming은 마치 사람이 타이핑을 치듯이 응답을 실시간으로 “흘려보내는” 방식이라고 이해할 수 있다.

- `모델.invoke(input, config)` → 응답 데이터
    - 모델이 전체 응답을 모두 생성한 뒤, 최종 결과를 한 번에 반환하는 방식이다.
    - 배치 처리나 후처리가 중요한 경우에 적합하다.
- `모델.stream(input, config)` → generator
    - 모델이 토큰을 생성하는 즉시, 순차적으로 제공하는 generator 를 반환한다.
    - 실시간 출력, 대화형 인터페이스, 웹 스트리밍 등에 특히 적합하다.

In [24]:
model = ChatOpenAI(model="gpt-5.4-nano")
res = model.invoke("서울에 유명한 여행지 세곳을 소개해주세요. 간단한 소개도 해주세요.")
print(res.content)

물론입니다! 서울에서 특히 유명한 여행지 3곳을 간단히 소개해드릴게요.

1) **경복궁**
- 조선 왕조의 대표적인 궁궐로, 웅장한 건축과 아름다운 정원이 함께 어우러진 곳이에요.  
- 특히 **수문장 교대의식** 같은 볼거리가 인기입니다.

2) **명동(명동거리)**
- 다양한 쇼핑과 먹거리로 유명한 서울의 대표적인 번화가예요.  
- 길거리 간식부터 백화점, 화장품 매장까지 한 번에 즐기기 좋아요.

3) **N서울타워(남산타워)**
- 남산 정상에서 서울 전경을 한눈에 볼 수 있는 전망 명소예요.  
- 야경이 특히 아름다워서 저녁 시간에 방문하면 더욱 좋습니다.

원하시면 여행 스타일(데이트/가족/혼자, 야경 위주/역사 위주 등)에 맞춰 동선도 짜드릴까요?


In [28]:
gen = model.stream("서울에 유명한 여행지 세곳을 소개해주세요. 간단한 소개도 해주세요.")
print(gen)

<generator object BaseChatModel.stream at 0x000002934B0EE320>


In [29]:
for token in gen:
    print(token.content, end="")

물론이에요! 서울에서 특히 유명한 여행지 3곳을 간단히 소개해드릴게요.

1) **경복궁**
- 조선 왕조의 대표적인 궁궐로, 웅장한 건축물과 아름다운 정원이 특징이에요.  
- 시간을 맞추면 **수문장 교대의식**도 볼 수 있어요.

2) **명동(명동 거리)**
- 쇼핑과 먹거리로 유명한 번화가예요.  
- 길거리 음식, 화장품/패션 매장, 다양한 레스토랑까지 즐길거리가 많아요.

3) **남산서울타워(남산공원)**
- 서울의 전경을 한눈에 볼 수 있는 전망 명소예요.  
- 케이블카/도보로 올라갈 수 있고, 야경이 특히 유명해요.

원하시면 “가족/커플/혼자” 중 어떤 여행 스타일인지 알려주시면 그에 맞춰 코스도 짜드릴게요!